# IOAI — 2025 Lab Lab3 Pedestrian Detection (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
print('데이터(lab3.zip)는 노트북 셀이 공개 GCS 에서 자동 다운로드합니다.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Lab 3

This week we look at a new dataset below.

In [ ]:
!curl https://storage.googleapis.com/aiolympiadmy/ioai-2025-tsp/lab3.zip -o lab3.zip

In [ ]:
!unzip -nq lab3.zip

The above commands should create a `train/` folder and `test/` folder in this directory. Each folder contains `images/` and `labels/` subdirectories respectively.

In this dataset, each image is associated with a text file. In the text file, each row represents a single bounding box around a person in the image. Each bounding box is represented by 5 numbers: x_center, y_center, width, height, objectness. Objectness is always 1. All other dimensions are represented as fractions of image dimensions. e.g. x_center's actual pixel location needs to be multiplied by the width of the image.

If you still recall the network you worked with in Lab 2:
```python
class FCN(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, x):
        x = self.backbone(x)
        x = self.head(x)
        return x

backbone = ...
head = ...
model = FCN(backbone, head)
```

Here's an idea. Given that the backbone defined above will output a spatial feature map, we can iterate over each cell location in the spatial feature map, and predict if there is an object to detect in that cell (objectness). Notice that this is very similar to FCN segmentation where we were practically doing pixel-wise classification. However, we take this a bit further. If objectness in each cell is high enough, we predict an object within a bounding box as specified by our neural network.

## Another new network

Create a new class that is almost the exact same as `FCN` above, but specify the `head` to be a `nn.Conv2d` layer with kernel size 1. This layer should output 5 channels, corresponding to the 5 numbers in each row of our labels. This network will be architecturally simpler than the FCN of Lab 2!

_1 pt granted upon completion of network definition_

In [ ]:
# Your work here
import torch
import torch.nn as nn
from torchvision.models import resnet34, ResNet34_Weights

In [ ]:
class FCN(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, x):
        x = self.backbone(x)
        x = self.head(x)
        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
my_model = FCN(
    nn.Sequential(*list(resnet34(weights=ResNet34_Weights.DEFAULT).children())[:-2]),
    nn.Sequential(
        nn.Conv2d(512, 5, kernel_size=1),
        nn.Sigmoid()
    )
).to(device)

## Dataset and dataloaders

Now go ahead and build your Dataset and Dataloaders in Pytorch.

Note that the dataset should calculate and return x_offset and y_offset instead of x_center and y_center. If you leave x_center and y_center as is, you will force your network to also learn how to predict larger values of x_center and y_center with increasing values of x and y on the spatial feature map output by the backbone! You can resolve this by storing the offset between the bounding box center and the center of the cell of the spatial feature map your bounding box is in. I'll help you a little by providing an example:

```python
ipdb>  grid_size
8

ipdb>  grid_y
4

ipdb>  grid_y_min
0.5

ipdb>  y_center
0.5132275132275133

ipdb>  y_offset
0.013227513227513255
```

_1 pt granted upon successfully running the code below_,

```python
train_dataset = ...
test_dataset = ...
train_dataloader = DataLoader(train_dataset, ...)
test_dataloader = DataLoader(test_dataset, ...)
one_X, one_y = next(iter(test_dataset))
batch_X, batch_y = next(iter(test_dataloader))
```

_and demonstrating the following results_:
- `one_X.shape` = (3, im_h, im_w) where im_h and im_w are height and width of the image
- `one_y.shape` = (5, gy, gx) where gy and gx are height and width of the spatial feature map
- `batch_X.shape` = (B, 3, im_h, im_w) where B is batch size
- `batch_y.shape` = (B, 5, gy, gx)

Unless you've gone out of your way to resize your images to a shape other than a square, `im_h` should be equal to `im_w`, and `gy` should be equal to `gx`.

In [ ]:
# Your work here
import albumentations as A
import numpy as np
from functools import lru_cache
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from albumentations.pytorch import ToTensorV2

In [ ]:
class MyDataset(Dataset):
    def __init__(self, path, gh, gw, image_folder="images", label_folder="labels"):
        super().__init__()
        self.img_dir = Path(path) / image_folder
        self.images = sorted(self.img_dir.glob("*.png"))
        self.label_dir = Path(path) / label_folder
        self.labels = sorted(self.label_dir.glob("*.txt"))
        self.transforms = A.Compose([
            A.Resize(224, 224),
            A.Normalize(),
            ToTensorV2()
        ], bbox_params=A.BboxParams(format="yolo", label_fields=["labels"]))
        self.gh = gh
        self.gw = gw

    def __len__(self):
        return len(self.images)

    def __setitem__(self, key, value):
        self.__dict__[key] = value
        self.__getitem__.cache_clear()

    @lru_cache(maxsize=None)
    def __getitem__(self, idx):
        img = np.array(Image.open(self.images[idx]).convert("RGB"))
        im_h, im_w, _ = img.shape
        
        label = np.loadtxt(self.labels[idx], delimiter=" ")
        if label.ndim == 1:
            label = label[np.newaxis, ...]
            
        if self.transforms is not None:
            img, label, _ = self.transforms(image=img, bboxes=label[:, :-1], labels=label[:, -1]).values()

        grid_label = torch.zeros((5, self.gh, self.gw))
        for box in label:
            gy_gx, box = self._helper(box)
            grid_label[:, *gy_gx] = box
        
        return img, grid_label

    def _helper(self, box):
        x_center_norm, y_center_norm, bbw_norm, bbh_norm = box
        gy_size_norm, gx_size_norm = 1 / self.gh, 1 / self.gw
    
        gy_norm, gx_norm = int(y_center_norm // gy_size_norm), int(x_center_norm // gx_size_norm)
        y_offset_norm, x_offset_norm = y_center_norm % gy_size_norm, x_center_norm % gx_size_norm
        
        return (gy_norm, gx_norm), torch.tensor([x_offset_norm, y_offset_norm, bbw_norm, bbh_norm, 1])

    def __iter__(self):
        for idx in range(len(self)):
            yield self[idx]

In [ ]:
train_dataset = MyDataset("./train", 7, 7)
test_dataset = MyDataset("./test", 7, 7)

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=16)
test_dataloader = DataLoader(test_dataset, batch_size=4)

In [ ]:
one_X, one_y = next(iter(test_dataset))

In [ ]:
one_X.shape, one_y.shape

In [ ]:
batch_X, batch_y = next(iter(test_dataloader))

In [ ]:
batch_X.shape, batch_y.shape

## Visualization

Being able to see is important for diagnosing computer vision applications. Create a visualization function to plot an overlay of bounding boxes on their respective images. You can use this function template below.

```python
def plot_batch_predictions(imgs, outputs):
    # img should have shape (batch, 3, im_h, im_w)
    # outputs should have shape (batch, 5, gy, gx)
    ...
```

Will also show you some sample matplotlib code to save time:

```python
import numpy as np
import matplotlib.pyplot as plt

# This creates a matplotlib figure with 4 cols
# Note that axes is an array of individual `ax`
batch_size = 4
fig, axes = plt.subplots(ncols=batch_size, figsize=(20, 8))
axes = axes if isinstance(axes, np.ndarray) else [axes]  # Handle batch_size=1

# This is how to plot an image
# Note that imshow requires image dimensions to be (H, W, 3)
# while Pytorch works with image dimensions (3, H, W)!
mock_tensor = torch.rand(3, 128, 128)
mock_np = mock_tensor.permute(1, 2, 0).contiguous().numpy()
# Usually you would just use plt.imshow where plt will grab the latest ax
# When you have as many as 4 in this example, specify which ax to use
ax = axes[0]
ax.imshow(img_np)
    
# This is how to draw rectangles using matplotlib
xmin = int((x_center - width / 2) * im_width)
xmax = int((x_center + width / 2) * im_width)
rect = plt.Rectangle(
    (xmin, ymin), width, height, 
    linewidth=2, edgecolor="red", facecolor="none"
)
# Add the rectangle to the Axes
ax.add_patch(rect)
```

Remember that your network output is x_offset and y_offset, need to convert them back to x_center and y_center!

_1 pt granted upon plotting one batch of images and labels from `test_dataloader` using `plot_batch_predictions`_

In [ ]:
# Your work here
import matplotlib.pyplot as plt
from torchvision.ops import nms

In [ ]:
def unnormalize(arr, transpose=True):  # Undo normalization, useful for displaying images
    if transpose:
        arr = np.transpose(arr, (1, 2, 0))

    # mean and std from ImageNet
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    arr = arr * (std[np.newaxis, np.newaxis, :] * 255) + (mean[np.newaxis, np.newaxis, :] * 255)
    return arr.astype(np.uint8)

In [ ]:
def to_pascal_voc_format(boxes, im_h, im_w):
    b, _, gh, gw = boxes.shape
    x_offset_norm, y_offset_norm, bbw_norm, bbh_norm, objectness = boxes.unbind(dim=1)

    gx = torch.arange(gw, device=boxes.device).reshape(1, 1, gw).expand(b, gh, gw)
    gy = torch.arange(gh, device=boxes.device).reshape(1, gh, 1).expand(b, gh, gw)
    y_center_norm, x_center_norm = y_offset_norm + gy.float() / gh, x_offset_norm + gx.float() / gw
    
    y_center, x_center = y_center_norm * im_h, x_center_norm * im_w
    bbh, bbw = bbh_norm * im_h, bbw_norm * im_w

    x_min, x_max = x_center - bbw / 2, x_center + bbw / 2
    y_min, y_max = y_center - bbh / 2, y_center + bbh / 2
    
    return torch.stack([x_min, y_min, x_max, y_max, objectness], dim=-1).permute(0, 3, 1, 2)

In [ ]:
def plot_batch_predictions(imgs, outputs, obj_threshold=0.5, nms_threshold=0.3):
    batch_size = imgs.shape[0]
    fig, axes = plt.subplots(ncols=batch_size, figsize=(5 * batch_size, 8))
    axes = axes if isinstance(axes, np.ndarray) else [axes]  # Handle batch_size=1

    batch_size, _, im_h, im_w = imgs.shape
    _, _, gh, gw = outputs.shape
    all_boxes = to_pascal_voc_format(outputs, im_h, im_w).permute(0, 2, 3, 1).reshape(batch_size, -1, 5)

    for idx, (img, boxes) in enumerate(zip(imgs, all_boxes)):
        ax = axes[idx]
        ax.imshow(unnormalize(img.numpy()))

        boxes = boxes[boxes[..., -1] >= obj_threshold]
        keep_indices = nms(boxes[..., :-1], boxes[..., -1], nms_threshold)
        boxes = boxes[keep_indices]

        for box in boxes:
            bbx_min, bby_min, bbx_max, bby_max, objectness = box
            bbh, bbw = bby_max - bby_min, bbx_max - bbx_min

            rect = plt.Rectangle(
                (bbx_min, bby_min), bbw, bbh,
                linewidth=2, edgecolor="red", facecolor="none"
            )
            ax.add_patch(rect)
    
    return fig

In [ ]:
plot_batch_predictions(batch_X, batch_y);

## Setting up loss calculations

Objectness is essentially a binary classification task, while predicting the correct bounding box is a regression task. 

Set up loss function calculations for objectness using binary cross entropy, and for bounding box localization using MSE. Create both losses in the same function for convenience, but return them separately instead of as a sum so they are easy to log later on. 

Concept-wise this is pretty straightforward. However, implementation-wise, you will need to place your tensors with great care. I'll help you a bit by providing you with this template below.

```python
def custom_loss(preds, targets):
    # both preds and targets should have shape (B, 5, gy, gx)
    # where B is batch size, gy and gx are spatial feature map h and w 
    ...
    return objectness_loss, localization_loss
```

_1 pt granted upon completion of loss function calculation_

In [ ]:
import torch.nn.functional as F

In [ ]:
def custom_loss(preds, targets):
    preds_objectness, targets_objectness = preds[:, -1, :, :], targets[:, -1, :, :]
    objectness_loss = F.binary_cross_entropy(preds_objectness, targets_objectness)
    
    has_box_mask = targets_objectness.unsqueeze(1).expand(-1, 4, -1, -1)
    num_true_boxes = has_box_mask.sum().item()
    if num_true_boxes > 0:
        localization_loss = F.mse_loss(preds[:, :-1, :, :] * has_box_mask,
                                       targets[:, :-1, :, :] * has_box_mask, reduction="sum")
        localization_loss /= num_true_boxes
    else:
        localization_loss = torch.tensor(0., device=preds.device)
        
    return objectness_loss, localization_loss

## Model evaluation and baseline score

Create a `test_one_epoch` function that takes the model and the test dataloader as arguments. Calculate and return box IOU score (`torchvision.ops.box_iou`) in a dictionary like so:

```pycon
>>> metrics = test_one_epoch(model, test_dataloader)
>>> print(metrics)
{"miou": 0.005}
```

_1 pt granted upon implementing `test_one_epoch` and seeing the mean IOU score of the untrained model_

In [ ]:
# Your work here
from torchvision.ops import box_iou
from scipy.optimize import linear_sum_assignment

In [ ]:
@torch.no_grad()
def test_one_epoch(model, dataloader, threshold=0.5):
    model.eval()
    score = 0
    count = 0
    for batch_images, batch_labels in dataloader:
        batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)

        batch_probas = model(batch_images)
        
        batch_size, _, im_h, im_w = batch_images.shape
        _, _, gh, gw = batch_probas.shape

        batch_pred_boxes = to_pascal_voc_format(batch_probas, im_h, im_w).permute(0, 2, 3, 1).reshape(batch_size, -1, 5)
        batch_truth_boxes = to_pascal_voc_format(batch_labels, im_h, im_w).permute(0, 2, 3, 1).reshape(batch_size, -1, 5)

        for single_preds, single_truths in zip(batch_pred_boxes, batch_truth_boxes):  # Process one image at a time
            has_truth = single_truths[..., -1] == 1
            has_pred = single_preds[..., -1] >= threshold
            count += has_truth.sum().item()

            iou_matrix = box_iou(single_preds[has_pred][..., :-1], single_truths[has_truth][..., :-1])
            # Use Hungarian algorithm to find optimal assignment that maximizes total IoU score
            # Each predicted bounding box is assigned to at most 1 ground-truth bounding box
            score += iou_matrix[*linear_sum_assignment(iou_matrix.cpu(), maximize=True)].sum().item()

    return {"miou": score / count}

In [ ]:
test_one_epoch(my_model, test_dataloader)

## Model training

Train your model on the training set. Track objectness loss and localization loss during training for every 10 minibatches (a). I will leave it up to choose how to combine your losses. 

At the end of every epoch, show metrics on both train (b) and test data (c), and plot prediction outputs of the first batch of the test dataset (d). Save the best performing model with the highest mean IOU score on test (e).

You don't need to run training for too long. I suspect <50 epochs will be sufficient.

_1 pt granted upon completion of (a) to (e)._

_Another 1 pt granted for exceeding 0.4 mean IOU on the test dataset._

In [ ]:
# Your work here
import os
import random
import torch.optim as optim

In [ ]:
def train(model, train_dl, test_dl, num_epochs, learning_rate, threshold=0.5, save_model_filename="best_model.pth"):
    train_loss_history = []
    highest = -1
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    for epoch in range(1, num_epochs + 1):
        for batch_idx, (batch_images, batch_labels) in enumerate(train_dl, 1):
            batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
            
            model.train()
            batch_probas = model(batch_images)
            batch_loss = custom_loss(batch_probas, batch_labels)
            # (a)
            if (epoch * batch_idx) % 10 == 0:
                train_loss_history.append(tuple(map(lambda x: x.item(), batch_loss)))
            batch_loss = sum(batch_loss)

            optimizer.zero_grad()
            batch_loss.backward()
            optimizer.step()

        # (b), (c)
        train_metric = test_one_epoch(model, train_dl, threshold)["miou"]
        test_metric = test_one_epoch(model, test_dl, threshold)["miou"]
        print(f"[Epoch {epoch}] {train_metric=:.6f}, {test_metric=:.6f}.")
        
        # (d)
        imgs_test, labels_test = next(iter(test_dl))
        with torch.no_grad():
            preds_test = model(imgs_test.to(device)).cpu()
        plot_batch_predictions(imgs_test, preds_test, threshold)
        
        # (e)
        if test_metric > highest:
            highest = test_metric
            torch.save(model.state_dict(), save_model_filename)
        
    return train_loss_history

In [ ]:
history = train(my_model, train_dataloader, test_dataloader, 12, 0.001)

In [ ]:
plt.plot(list(map(lambda x: x[0], history)), label="BCE")
plt.plot(list(map(lambda x: x[1], history)), label="MSE")
plt.plot(list(map(sum, history)), label="Custom")
plt.legend()
plt.title("Training Loss")
plt.xlabel("minibatch")
plt.ylabel("loss");

## Post training

Create a plot that contains four subplots: image with true bounding boxes, image with predicted bounding boxes, predicted objectness over spatial feature map, true objectness over spatial feature map. Repeat this plot for a few images.

_1 pt granted upon completion of the above_

In [ ]:
# Your work here
import seaborn as sns
import matplotlib.colors as mcolors
import matplotlib.patches as patches
from copy import deepcopy

In [ ]:
best_model = deepcopy(my_model)
best_model.load_state_dict(torch.load("best_model.pth"))

In [ ]:
# Your work here
def post_training_plot(img, pred, label, threshold=0.5):
    fig = plot_batch_predictions(torch.stack([img, img, img, img]), torch.stack([label, pred, pred, pred]), threshold)  # third and fourth are dummies

    fig.axes[0].set_title("True Bounding Boxes")
    fig.axes[1].set_title("Predicted Bounding Boxes")

    for ax_idx in range(2):
        ax = fig.axes[ax_idx]
        ax.grid(True, which="both", axis="both", color="white", linestyle="-", linewidth=0.5)
        ax.set_xticks(np.arange(0, img.shape[2] + 1, img.shape[2] / label.shape[2]))
        ax.set_yticks(np.arange(0, img.shape[1] + 1, img.shape[1] / label.shape[1]))

        for patch in ax.patches:
            if not isinstance(patch, patches.Rectangle):
                continue

            x, y = patch.get_xy()
            width, height = patch.get_width(), patch.get_height()

            x_center, y_center = x + width / 2, y + height / 2
            ax.plot(x_center, y_center, "ro")

    cmap = mcolors.ListedColormap(["black", "#2A2A2A", "white"])
    bounds = [0, max(0, threshold - 0.1), threshold, 1]
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    ax2 = fig.axes[2]
    ax2.clear()
    sns.heatmap(label[-1, :, :], ax=ax2, cbar=False, annot=True, cmap=cmap, norm=norm, fmt=".2f")
    ax2.set_title("True Objectness")

    ax3 = fig.axes[3]
    ax3.clear()
    sns.heatmap(pred[-1, :, :], ax=ax3, cbar=False, annot=True, cmap=cmap, norm=norm, fmt=".2f")
    ax3.set_title("Predicted Objectness")
    
    return fig

In [ ]:
tmp = iter(test_dataloader)
next(tmp)
imgs_to_show, labels_to_show = next(tmp)
with torch.no_grad():
    preds_to_show = best_model(imgs_to_show.to(device)).cpu()

for i in range(4):
    post_training_plot(imgs_to_show[i], preds_to_show[i], labels_to_show[i]);

What learning task did you just perform in this notebook?

_1 pt granted upon finding the right answer_

\# Your answer here

A YOLO-style object detection.

This dataset is 18 years old this year. What is it called?

_1 pt granted upon finding the right answer_

\# Your answer here

[Penn-Fudan Database for Pedestrian Detection and Segmentation.](https://www.cis.upenn.edu/~jshi/ped_html/)

## EX: Going off track

Name a limitation of this training setup and briefly explain your reasoning.

_1 pt granted upon a satisfactory answer_

\# Your work here

One limitation is that each grid cell can predict at most one box. If the centers of two people fall into the same cell, the model can only ever detect one of them, so it will systematically miss objects in crowded scenes.

---

Show me how you can modify this training setup to attain better performance.

_2 pts granted upon successfully scoring at least +0.2 mean IOU higher than the score of the best model above. Partial credit to be granted at discretion. Bonus additional +1 pt to be granted for outstanding improvements_

In [ ]:
# Your work here
def ciou(preds, targets, eps=1e-7):  # Expect shape (num_boxes, 5)
    pred_x_min, pred_y_min, pred_x_max, pred_y_max, _ = preds.unbind(dim=1)
    truth_x_min, truth_y_min, truth_x_max, truth_y_max, _ = targets.unbind(dim=1)

    # Compute IoU
    pred_area = (pred_x_max - pred_x_min) * (pred_y_max - pred_y_min)
    truth_area = (truth_x_max - truth_x_min) * (truth_y_max - truth_y_min)

    inter_x_min, inter_y_min = torch.max(pred_x_min, truth_x_min), torch.max(pred_y_min, truth_y_min)
    inter_x_max, inter_y_max = torch.min(pred_x_max, truth_x_max), torch.min(pred_y_max, truth_y_max)
    
    inter_area = torch.clamp(inter_x_max - inter_x_min, min=0) * torch.clamp(inter_y_max - inter_y_min, min=0)
    union_area = pred_area + truth_area - inter_area
    iou = inter_area / (union_area + eps)

    # Compute Euclidean distance between centers
    pred_x_center, pred_y_center = (pred_x_min + pred_x_max) / 2, (pred_y_min + pred_y_max) / 2
    truth_x_center, truth_y_center = (truth_x_min + truth_x_max) / 2, (truth_y_min + truth_y_max) / 2

    center_dist_squared = (pred_x_center - truth_x_center) ** 2 + (pred_y_center - truth_y_center) ** 2  # rho^2

    # Compute the diagonal length of the smallest enclosing box
    enclosing_x_min, enclosing_y_min = torch.min(pred_x_min, truth_x_min), torch.min(pred_y_min, truth_y_min)
    enclosing_x_max, enclosing_y_max = torch.max(pred_x_max, truth_x_max), torch.max(pred_y_max, truth_y_max)
    
    enclosing_diag_squared = (enclosing_x_max - enclosing_x_min) ** 2 + (enclosing_y_max - enclosing_y_min) ** 2 + eps  # c^2

    # Compute aspect ratio penalty
    pred_width, pred_height = torch.clamp(pred_x_max - pred_x_min, min=0), torch.clamp(pred_y_max - pred_y_min, min=0)
    truth_width, truth_height = torch.clamp(truth_x_max - truth_x_min, min=0), torch.clamp(truth_y_max - truth_y_min, min=0)

    v = (4 / (torch.pi ** 2)) * (torch.atan(truth_width / (truth_height + eps)) - torch.atan(pred_width / (pred_height + eps))) ** 2
    with torch.no_grad():
        alpha = v / (1 - iou + v + eps)

    # Final CIoU
    ciou = iou - (center_dist_squared / enclosing_diag_squared) - alpha * v
    return ciou

$$
\text{CIoU} = \text{IoU} - \frac{\rho^2(\mathbf b - \mathbf b^{gt})}{c^2} - \alpha v
$$

where:

- $\mathbf b$: predicted bounding box.
- $\mathbf b^{gt}$: ground-truth bounding box.
- $\rho(\cdot)$: Euclidean distance between the centers.
- $c$: diagonal length of the smallest enclosing box.
- $v = \frac{4}{\pi^2} (\arctan \frac{w^{gt}}{h^{gt}} - \arctan \frac{w}{h})^2$.
- $\alpha = \frac{v}{(1 - \text{IoU})+v}$.
- $w$, $h$: width and height of the predicted bounding box.
- $w^{gt}$, $h^{gt}$: width and height of the ground-truth bounding box.

In [ ]:
def custom_loss(preds, targets, im_h=224, im_w=224):
    preds_objectness, targets_objectness = preds[:, -1, :, :], targets[:, -1, :, :]
    objectness_loss = F.binary_cross_entropy(preds_objectness, targets_objectness)

    pred_boxes = to_pascal_voc_format(preds, im_h, im_w).permute(0, 2, 3, 1).reshape(-1, 5)
    truth_boxes = to_pascal_voc_format(targets, im_h, im_w).permute(0, 2, 3, 1).reshape(-1, 5)

    has_truth = truth_boxes[:, -1] == 1
    ciou_loss = (1 - ciou(pred_boxes[has_truth], truth_boxes[has_truth])).mean()

    return objectness_loss, ciou_loss

In [ ]:
train_dataset_aug = MyDataset("./train", 7, 7)
train_dataset_aug.transforms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),

    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),

    A.MotionBlur(p=0.4),
    A.GaussNoise(std_range=(0.05, 0.1), p=0.2),

    A.CLAHE(p=0.1),
    A.RandomBrightnessContrast(p=0.3),

    A.Normalize(),
    ToTensorV2()
], bbox_params=A.BboxParams(format="yolo", label_fields=["labels"]))
train_dataloader_aug = DataLoader(train_dataset_aug, batch_size=16)

In [ ]:
batch_X_aug, batch_y_aug = next(iter(train_dataloader_aug))
plot_batch_predictions(batch_X_aug[:4], batch_y_aug[:4]);

In [ ]:
train(my_model, train_dataloader_aug, test_dataloader, 49, 0.05, threshold=0.1, save_model_filename="best_model_ex.pth");

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv', 'submission.zip', 'submission.jsonl', 'submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)